# MethylGPT quickstart

Predict age and 25 full-array clock outputs, convert a 27K-array profile to the full 49,156-CpG vocabulary, and compute sample embeddings. The example file holds six public 450K profiles from GSE54848 (dermal fibroblasts and iPSCs after three passages); none of them were used to train MethylGPT.

Set `BUNDLE` to the unpacked model bundle (`methylgpt-1.1`). Predicted age is the mean of three training runs. A GPU is used when available; CPU also works.

In [ ]:
import os
import numpy as np
import pandas as pd
from methylgpt import MethylGPT

BUNDLE = os.environ.get('METHYLGPT_BUNDLE', 'methylgpt-1.1')
model = MethylGPT.from_pretrained(BUNDLE)
print(model.device, len(model.vocab), 'vocabulary CpGs')

## Input
Rows are samples, columns are CpG probe IDs, values are beta values in [0, 1]. Missing CpGs can be absent or NaN; no imputation is needed beforehand.

In [ ]:
betas = pd.read_csv('GSE54848_example_betas.csv', index_col=0)
betas.iloc[:, :5]

## Age, sex, tissue and distilled clocks

In [ ]:
pred = model.predict(betas)
pred[['vocab_cpgs_observed', 'age', 'p_male', 'tissue', 'GrimAge2', 'DunedinPACE', 'PhenoAge']].round(2)

## Array conversion: keep only CpGs on the 27K array, impute the rest
The 27K array covers about 45% of the vocabulary. Here the other CpGs are removed and then predicted, and the predictions are compared with the measured values.

In [ ]:
on27k = set(pd.read_csv('hm27_vocab_cpgs.txt', header=None)[0])
reduced = betas[[c for c in betas.columns if c in on27k]]
full = model.impute(reduced)
held = [c for c in betas.columns if c not in on27k]
err = (full[held] - betas[held]).abs()
print('kept', reduced.shape[1], 'CpGs; predicted', len(held), 'measured CpGs; mean absolute error',
      round(float(np.nanmean(err.to_numpy())), 4))

In [ ]:
pd.DataFrame({'age, all CpGs': pred.age, 'age, 27K CpGs only': model.predict_age(reduced)}).round(1)

## Reconstruction and residuals
`reconstruct()` returns the factor model's reconstruction of every vocabulary CpG, including the measured ones; `residuals()` returns the measured value minus this reconstruction (NaN where a CpG is not measured). `impute()`, used above, keeps measured values and fills only the others.

In [ ]:
rec = model.reconstruct(betas)
res = model.residuals(betas)
print(rec.shape, res.shape)
print('mean absolute residual at measured CpGs:', round(float(np.nanmean(np.abs(res.to_numpy()))), 4))
print('unmeasured CpGs in the first profile:', int(res.iloc[0].isna().sum()))

## Sample embeddings
A 2,048-dimensional representation (standardised factor scores plus pooled transformer latents) that is the input of the prediction heads; use it for downstream models, and `factor_scores()` (below) for clustering.

In [ ]:
emb = model.embed(betas)
print(emb.shape)
np.round(np.corrcoef(emb.to_numpy())[:3, :3], 3)

## Factor scores and the features of the paper's disease models
`factor_scores()` gives the 1,536 factor scores (use these for clustering and nearest-neighbour search). `embed()` returns the embedding of the released encoder; the disease models of the paper were fitted on the embedding of an earlier checkpoint, returned by `embed(stage='age_stage')`, and `disease_features()` returns their complete feature set (that embedding, the 25 clock outputs in standardized units and the 49,156 factor-model residuals).

In [ ]:
fs = model.factor_scores(betas)
E = model.embed(betas, stage='age_stage')
F = model.disease_features(betas)
print(fs.shape, E.shape, F.shape)
print('correlation of the two embeddings, first profile:', round(float(np.corrcoef(E.iloc[0], emb.iloc[0])[0, 1]), 3))